# Hybrid Dynamic Ensemble for Mag-7 Stock Return Prediction
## Complete Implementation Pipeline 

This notebook presents the complete research pipeline developed to forecast daily returns of the Magnificent Seven stocks. The central contribution is a Hybrid Dynamic Ensemble (HDE) that adaptively combines classical machine learning models with deep learning sequence models, allocating weights based on each model's empirical performance on held-out validation data.

**Pipeline Overview:**
1. Data Collection (Yahoo Finance + FRED)


## Script 01: S&P 500 / Mag-7 Stock Price Data Collection

In [4]:
import yfinance as yf
import os
import pandas as pd

# Configuration for the "Research Baseline"
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'SPY']
START_DATE = "2015-01-01"
END_DATE = "2026-01-01"  # Ensures data stops at 2025-12-31
OUTPUT_PATH = "data/raw/prices.csv"

def collect_prices():
    print(f"Downloading Mag 7 + Benchmark: {START_DATE} to {END_DATE}")
    
    # auto_adjust=True is critical for dividends/splits in dissertation research
    df = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)
    
    # yfinance returns multi-level columns, just need Close prices
    if isinstance(df.columns, pd.MultiIndex):
        data = df['Close']
    else:
        data = df
        
    data.index.name = 'Date'
    
    # Save the data
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    data.to_csv(OUTPUT_PATH)
    
    print(f"Saved {len(data)} rows to {OUTPUT_PATH}")
    print(f"Date range: {data.index.min()} to {data.index.max()}")
    print(f"Tickers: {list(data.columns)}")
    return data

if __name__ == "__main__":
    collect_prices()

[*********************100%***********************]  8 of 8 completed

Saved 2766 rows to data/raw/prices.csv
Date range: 2015-01-02 00:00:00 to 2025-12-31 00:00:00
Tickers: ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'SPY', 'TSLA']


## Script 02 - Federal Reserve / Macroeconomic Data Collection

In [ ]:
import pandas as pd
from fredapi import Fred
import os
from dotenv import load_dotenv

load_dotenv()

FRED_API_KEY = os.environ.get('FRED_API_KEY')
OUTPUT_PATH = 'data/raw/macro_fred.csv'
START_DATE = '2014-01-01'
END_DATE = '2026-12-31'

MACRO_SERIES = {
    'fed_funds_rate': 'FEDFUNDS',
    'cpi': 'CPIAUCSL',
    'unemployment': 'UNRATE',
    'gdp_growth': 'A191RL1Q225SBEA',
    'vix': 'VIXCLS',
    'treasury_10y': 'GS10',
    'treasury_2y': 'GS2'
}